# Cierre de curso: recomendador de Steam, propio vs. preentrenado

**La despedida.** Junta las dos últimas sesiones — `PySpark_Recommenders` (ALS) y
`PySpark_NLP_Steam` (el dataset, la limpieza, el modelo propio de sentimiento) — sobre el
mismo archivo que ya exploraron: `steam_reviews.csv`. Mismo dato, pregunta distinta: la
sesión de texto preguntaba *"¿el texto predice si recomienda?"*; hoy preguntamos
*"¿con cuánto tiempo jugó y si lo recomendó, qué otro juego le sugerimos — y qué tan
buena es la señal de un modelo que nunca vio Steam, comparada con el que ustedes
entrenaron?"*

## Lo nuevo de hoy (el "bang")

1. **ALS con feedback implícito**, no explícito. `steam_reviews.csv` no trae una
   calificación 1-10 como el dataset de anime — solo `recommended` (sí/no) y
   `author.playtime_forever` (minutos jugados). Combinamos ambas en una sola señal de
   *confianza* en vez de usar una sola:
   `engagement = log1p(minutos_jugados) × (BOOST si recomienda, DAMPEN si no)`.
   Dos personas con el mismo tiempo jugado no están diciendo lo mismo si una lo
   recomienda y la otra no — esa asimetría es información, no ruido, y la regla estándar
   de ALS implícito (`c_ui = 1 + α·r_ui`) no la captura si solo le das el playtime.
2. **La métrica correcta para feedback implícito.** RMSE (lo que usó `Recommenders`)
   no aplica aquí — no hay una calificación real contra la cual medir el error. Se usa
   `RankingMetrics` (Precision@K, MAP, NDCG@K): ¿las recomendaciones del top-K
   coinciden con los juegos de más *engagement* real que esa persona tuvo en test?
3. **Propio vs. preentrenado.** El `PipelineModel` que guardaron en `NLP_Steam`
   (TF-IDF + regresión logística, entrenado por ustedes) contra
   `distilbert-base-uncased-finetuned-sst-2-english` de Hugging Face (67M parámetros,
   entrenado por otros, nunca vio una reseña de videojuego) — sobre las reseñas de los
   juegos que el recomendador sugirió. ¿Coinciden?

## ⚠️ Lo primero que hay que verificar — puede tumbar todo lo demás

**ALS necesita un identificador de usuario estable.** Las columnas que ya conocíamos de
`steam_reviews.csv` (`app_id`, `app_name`, `review`, `recommended`, `author.*`) no
confirman una — pero el esquema real de la API de Steam (de donde sale este dataset) sí
incluye `author.steamid`. La celda de abajo lo busca explícitamente y **se detiene con un
mensaje claro** si no aparece, en vez de inventar un ID falso. Si tu copia del CSV
de verdad no lo trae, este notebook no se puede correr tal cual — avisa antes de usar
cluster.

## Cluster: sección B.1 de `MLOPS/README.md`

Necesita `transformers`/`torch` (ya instalados ahí vía `hugging_face_deps.sh`) para el
bloque de Hugging Face — el resto (ALS, limpieza) corre igual en B. **Corregido en este
PR:** el README decía que los workers `n1-standard-4` tienen "30 GB de RAM, para usar el
modelo BART" — la especificación oficial de GCP es **15 GB** por worker (4 vCPU). Sigue
alcanzando de sobra para DistilBERT (~300 MB cargado), que es justo el modelo que usamos
aquí — pero no asumas que cabe cualquier cosa más grande sin volver a verificar.

**Sin GPU: la inferencia de Hugging Face corre en CPU.** Probado en este entorno:
~95 reseñas/segundo de un solo hilo, sin *batching* fino — suficiente para las reseñas de
un puñado de juegos recomendados, no para el archivo completo. Por eso `MAX_RESENAS_HF`
acota la muestra, con la misma disciplina de `FRACCION`/`MAX_ROWS` que ya usan los otros
notebooks.

**Requiere `HF_TOKEN`.** El modelo (`distilbert-base-uncased-finetuned-sst-2-english`) es
público — correría sin token — pero este notebook lo exige de todas formas. Es una
variable explícita en la celda de parámetros (`HF_TOKEN = "<TU_TOKEN_DE_HUGGING_FACE>"`,
mismo patrón que `BUCKET`) — **pégalo ahí**. Como este repo es público, la celda avisa:
si vas a commitear el notebook después de correrlo, vuelve a dejar el placeholder antes de
guardar. Si prefieres no tocar el archivo, la misma celda también acepta una variable de
entorno o la metadata de la instancia de GCE como respaldo.

In [ ]:
# Parche de compatibilidad para el Jupyter de Dataproc: matplotlib llama a RcParams._get,
# que no existe en este entorno. Debe ejecutarse ANTES de importar pyplot.
import matplotlib
_RcParams = type(matplotlib.rcParams)
if not hasattr(_RcParams, '_get'):
    _RcParams._get = _RcParams.get

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, StringType, StructType, StructField
from pyspark.sql.window import Window
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS
from pyspark.ml.functions import vector_to_array
from pyspark.mllib.evaluation import RankingMetrics

In [ ]:
spark = SparkSession.builder.appName("Recomendador_Steam_Cierre").getOrCreate()

In [ ]:
# --- Ajustar a tu bucket ---
BUCKET = "big-data-lunes-20260223"
RUTA_CSV     = f"gs://{BUCKET}/steam/steam_reviews.csv"
RUTA_PARQUET = f"gs://{BUCKET}/steam/steam_reviews_parquet"   # lo crea NLP_Steam, o esta misma celda
RUTA_MODELO_PROPIO = f"gs://{BUCKET}/modelos/steam_nlp"       # el PipelineModel que guardó NLP_Steam — este SÍ tiene que existir de antes

# False si ya corriste NLP_Steam con CONVERTIR_A_PARQUET=True sobre el mismo bucket (lo normal).
# True si este es tu primer notebook tocando steam_reviews.csv, o si cambiaste de bucket.
CONVERTIR_A_PARQUET = False

FRACCION = 0.1              # sube conforme veas tiempos; ALS es más caro que TF-IDF
MIN_RESENAS_JUEGO = 50      # mismo filtro de popularidad que Recommenders
BOOST, DAMPEN = 1.5, 0.5    # peso del engagement si recomienda / si no
ALPHA_ALS = 40.0            # valor típico citado en Hu/Koren/Volinsky (2008) — no ajustado a este dataset, pruébalo

USAR_HUGGINGFACE = True     # False corre todo menos el bloque de comparación de modelos
MAX_RESENAS_HF = 500        # tope duro — CPU sin GPU, no correr sobre el archivo completo

In [ ]:
# --- Token de Hugging Face: obligatorio si USAR_HUGGINGFACE=True ---
# Token gratis (de lectura basta) en https://huggingface.co/settings/tokens. Pégalo abajo.
#
# ⚠️ Este repo es público en GitHub. Si vas a hacer commit/push de este notebook después
# de correrlo, vuelve a dejar el placeholder antes de guardar — un token real commiteado
# queda en el historial de git para siempre, aunque lo borres en un commit posterior.
HF_TOKEN = "<TU_TOKEN_DE_HUGGING_FACE>"

# Respaldo opcional, por si prefieres no tocar esta celda: si arriba se queda el
# placeholder, se intenta la variable de entorno y luego la metadata de la instancia de
# GCE (gcloud compute instances add-metadata <CLUSTER>-m --metadata=HF_TOKEN=...) — mismo
# mecanismo que ya usa este repo en Synthetic_Data/deploy/gcp/run_pilot.py. Ninguna de las
# dos vías dejaría el token en el archivo.
if HF_TOKEN == "<TU_TOKEN_DE_HUGGING_FACE>" or not HF_TOKEN.strip():
    import os
    from urllib.request import Request, urlopen
    from urllib.error import URLError, HTTPError

    def _leer_metadata_gcp(atributo, timeout=2):
        url = f"http://metadata.google.internal/computeMetadata/v1/instance/attributes/{atributo}"
        try:
            with urlopen(Request(url, headers={"Metadata-Flavor": "Google"}), timeout=timeout) as resp:
                return resp.read().decode().strip()
        except (URLError, HTTPError, OSError):
            return None

    HF_TOKEN = os.environ.get("HF_TOKEN", "").strip() or (_leer_metadata_gcp("HF_TOKEN") or "")

if USAR_HUGGINGFACE:
    assert HF_TOKEN, (
        "USAR_HUGGINGFACE=True pero no hay HF_TOKEN. Pégalo directo en la variable de "
        "arriba, o déjala en el placeholder y pásalo por variable de entorno o metadata "
        "del cluster. O pon USAR_HUGGINGFACE=False para correr todo menos el bloque de "
        "comparación de modelos."
    )
    print("HF_TOKEN detectado — el bloque de Hugging Face usará autenticación.")

## 1. Cargar y verificar que hay un ID de usuario

Mismo patrón de ingesta que `NLP_Steam` (CSV multilínea → Parquet una vez → releer). La
diferencia es la verificación de `steamid`.

In [ ]:
if CONVERTIR_A_PARQUET:
    # Mismo patrón que NLP_Steam: CSV multilínea (las reseñas traen saltos de línea
    # dentro de comillas) -> Parquet una vez -> se relee siempre desde Parquet.
    raw_csv = spark.read.csv(RUTA_CSV, header=True, multiLine=True, escape='"', quote='"')
    raw_csv = raw_csv.toDF(*[c.replace(".", "_") for c in raw_csv.columns])
    raw_csv.write.mode("overwrite").parquet(RUTA_PARQUET)

try:
    raw = spark.read.parquet(RUTA_PARQUET)
except Exception as e:
    raise AssertionError(
        f"No se pudo leer {RUTA_PARQUET}. O corre NLP_Steam primero (con "
        "CONVERTIR_A_PARQUET=True al menos una vez), o pon CONVERTIR_A_PARQUET=True "
        "aquí arriba para generarlo directo del CSV."
    ) from e

if FRACCION < 1.0:
    raw = raw.sample(fraction=FRACCION, seed=42)

candidatos_id = ["author_steamid", "author_id", "steamid"]
col_usuario = next((c for c in candidatos_id if c in raw.columns), None)
assert col_usuario is not None, (
    f"No se encontró columna de usuario entre {candidatos_id}. "
    f"Columnas disponibles: {sorted(raw.columns)}. "
    "Sin un ID de usuario estable no se puede construir ALS sobre este archivo."
)
print(f"Columna de usuario encontrada: '{col_usuario}'")
raw = raw.withColumnRenamed(col_usuario, "author_id")
raw.printSchema()

## 2. Limpieza (igual que `NLP_Steam`, condensada) + verificar `playtime_forever`

Antes de confiar en el playtime como señal: ¿es el tiempo jugado **en ese juego**, o un
contador global del usuario repetido en cada fila? Si fuera global, todas las reseñas del
mismo autor tendrían el mismo valor — lo comprobamos, no lo asumimos.

In [ ]:
cols = set(raw.columns)
faltan = {"review", "recommended", "app_id", "author_playtime_forever"} - cols
assert not faltan, f"Faltan columnas obligatorias {faltan}"

label_txt = F.lower(F.col("recommended").cast("string"))
df = (raw
      .withColumn("recomienda", F.when(label_txt.isin("true", "1"), True)
                                 .when(label_txt.isin("false", "0"), False))
      .withColumn("playtime", F.col("author_playtime_forever").cast(DoubleType()))
      .filter(col("recomienda").isNotNull() & col("review").isNotNull()
              & col("playtime").isNotNull() & (col("playtime") >= 0)
              & col("author_id").isNotNull() & col("app_id").isNotNull()))

if "language" in cols:
    df = df.filter(F.lower(col("language")) == "english")

df = df.dropDuplicates(["author_id", "app_id"])   # una señal por par (usuario, juego)
n = df.count()
print(f"reseñas utilizables: {n:,}")

In [ ]:
# Autores con >=3 reseñas: ¿varía el playtime entre juegos distintos, o es constante?
multi = (df.groupBy("author_id").agg(F.count("*").alias("n"), F.countDistinct("playtime").alias("valores_distintos"))
         .filter(col("n") >= 3))
pct_constante = multi.filter(col("valores_distintos") == 1).count() / max(multi.count(), 1)
print(f"Autores con >=3 reseñas: {multi.count():,}")
print(f"De esos, con el MISMO playtime en todas sus reseñas: {pct_constante:.1%}")
print("Si ese % es alto (cercano a 100%), 'playtime' es un contador global, no por juego — "
      "replantear la señal de engagement antes de seguir.")

## 3. El engagement: dos señales, una sola columna

`log1p` porque el playtime tiene una cola larguísima (unos pocos jugadores acumulan miles
de horas). `BOOST`/`DAMPEN` codifican la asimetría de recomendar o no.

In [ ]:
df_eng = (df
    .withColumn("engagement", F.log1p(col("playtime")) *
                F.when(col("recomienda"), F.lit(BOOST)).otherwise(F.lit(DAMPEN)))
    .select("author_id", "app_id", "app_name", "engagement", "recomienda", "playtime"))

df_eng.select("engagement").summary("min", "25%", "50%", "75%", "max").show()

## 4. IDs numéricos, split, y el filtro de popularidad de siempre

Mismo problema que ya vieron en `Recommenders`: ALS tiende a sobre-recomendar juegos con
pocas reseñas (una sola reseña con mucho playtime infla el factor latente). Se filtra
**antes** de entrenar, no después — así el modelo nunca aprende sobre ese ruido.

In [ ]:
soporte = df_eng.groupBy("app_id").count()
juegos_validos = soporte.filter(col("count") >= MIN_RESENAS_JUEGO).select("app_id")
df_eng = df_eng.join(juegos_validos, "app_id")
print(f"Juegos con >= {MIN_RESENAS_JUEGO} reseñas: {juegos_validos.count():,}")

In [ ]:
user_idx = StringIndexer(inputCol="author_id", outputCol="user_id").setHandleInvalid("skip")
item_idx = StringIndexer(inputCol="app_id", outputCol="item_id").setHandleInvalid("skip")
prep = Pipeline(stages=[user_idx, item_idx]).fit(df_eng)
df_final = prep.transform(df_eng)

train, test = df_final.randomSplit([0.8, 0.2], seed=42)

# BLOQUE ALS — feedback implícito

`implicitPrefs=True` cambia qué significa `ratingCol`: ya no es una calificación a
predecir, es la fuerza de una señal observada. Spark construye internamente
`confianza = 1 + alpha * engagement` — por eso `alpha` es aparte, no se mete a mano en la
columna.

In [ ]:
%%time
als = ALS(implicitPrefs=True, userCol="user_id", itemCol="item_id", ratingCol="engagement",
          alpha=ALPHA_ALS, rank=15, maxIter=10, regParam=0.1,
          coldStartStrategy="drop", seed=42)
model = als.fit(train)

## Evaluación: `RankingMetrics`, no RMSE

"Relevante" en test = engagement por encima del percentil 70 de train (jugó mucho Y/o lo
recomendó). Comparamos el top-K de ALS contra eso, y contra una línea base de "siempre
recomendar los juegos de mayor engagement total" — el mismo espíritu de línea base que ya
usaron en `Recommenders`, adaptado a ranking.

In [ ]:
K_EVAL = 10
recs = model.recommendForAllUsers(K_EVAL)
pred_als = recs.select("user_id", F.expr("transform(recommendations, x -> x.item_id)").alias("pred"))

umbral = train.approxQuantile("engagement", [0.7], 0.01)[0]
relevante = (test.filter(col("engagement") >= umbral)
             .groupBy("user_id").agg(F.collect_list("item_id").alias("actual")))

top_global = (train.groupBy("item_id").agg(F.sum("engagement").alias("total"))
              .orderBy(F.desc("total")).limit(K_EVAL)
              .select(F.collect_list("item_id")).first()[0])
pred_base = relevante.select("user_id").withColumn("pred", F.array([F.lit(float(x)) for x in top_global]))

def evaluar(nombre, pred_df):
    joined = pred_df.join(relevante, "user_id", "inner")
    rdd = joined.rdd.map(lambda r: ([float(x) for x in r["pred"]], [float(x) for x in r["actual"]]))
    m = RankingMetrics(rdd)
    print(f"[{nombre}] usuarios evaluados: {joined.count():,} | "
          f"MAP: {m.meanAveragePrecision:.4f} | Precision@{K_EVAL}: {m.precisionAt(K_EVAL):.4f} | "
          f"NDCG@{K_EVAL}: {m.ndcgAt(K_EVAL):.4f}")

evaluar("línea base: top global", pred_base)
evaluar("ALS implícito", pred_als)

**Cómo leerlo:** si ALS no le gana claramente a "recomendsatle a todos los juegos más
populares", el modelo no está aprendiendo preferencia personal — solo popularidad, que ya
sabíamos de entrada. La pregunta de la clase pasada ("¿se concentran las recomendaciones
en pocos títulos?") aplica otra vez aquí.

# Recomendaciones concretas, para el bloque de comparación de modelos

Tomamos 2 usuarios con historial real (no inventado) para seguirlos hasta el final.

In [ ]:
mapping_juego = df_final.select("item_id", "app_id", "app_name").distinct()
usuarios_ejemplo = [r["user_id"] for r in
                     df_final.groupBy("user_id").count().orderBy(F.desc("count")).limit(2).collect()]

recs_ejemplo = (recs.filter(col("user_id").isin(usuarios_ejemplo))
                 .withColumn("rec", F.explode("recommendations"))
                 .select("user_id", F.col("rec.item_id").alias("item_id"), F.col("rec.rating").alias("score"))
                 .join(mapping_juego, "item_id")
                 .select("user_id", "app_id", "app_name", "score"))
recs_ejemplo.orderBy("user_id", F.desc("score")).show(20, truncate=40)

# BLOQUE DE CIERRE — modelo propio vs. preentrenado

Reseñas reales de los juegos recomendados arriba (tope `MAX_RESENAS_HF`, nunca el archivo
completo). Dos jueces de sentimiento sobre el mismo texto: el que entrenaron en
`NLP_Steam` y uno que nunca vio Steam.

In [ ]:
app_ids_recomendados = [r["app_id"] for r in recs_ejemplo.select("app_id").distinct().collect()]
muestra_resenas = (df.filter(col("app_id").isin(app_ids_recomendados))
                    .select("app_id", "app_name", "review")
                    .limit(MAX_RESENAS_HF)
                    .persist())
print(f"Reseñas a comparar: {muestra_resenas.count():,} (de los {len(app_ids_recomendados)} juegos recomendados)")

In [ ]:
if USAR_HUGGINGFACE:
    def limpiar(d):
        return (d
                .withColumn("texto", F.regexp_replace(F.lower(col("review")), r"https?://\S+", " "))
                .withColumn("texto", F.regexp_replace("texto", r"\[/?[a-z0-9=*]+\]", " "))
                .withColumn("texto", F.regexp_replace("texto", r"\s+", " "))
                .filter(F.length("texto") >= 10))

    # --- Modelo propio: el PipelineModel guardado por NLP_Steam ---
    modelo_propio = PipelineModel.load(RUTA_MODELO_PROPIO)
    pred_propio = (modelo_propio.transform(limpiar(muestra_resenas))
                   .select("app_id", "review",
                           F.round(vector_to_array("probability")[1], 3).alias("p_propio"))
                   .withColumn("veredicto_propio", (col("p_propio") >= 0.5)))

In [ ]:
if USAR_HUGGINGFACE:
    schema_hf = StructType([StructField("hf_label", StringType()), StructField("hf_score", DoubleType())])

    @F.pandas_udf(schema_hf)
    def sentimiento_hf(textos: pd.Series) -> pd.DataFrame:
        # Se carga UNA vez por proceso de worker (variable global de módulo), no por fila ni por batch.
        global _CLF_HF
        try:
            _CLF_HF
        except NameError:
            from transformers import pipeline
            _CLF_HF = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", token=HF_TOKEN)
        out = _CLF_HF(textos.tolist(), truncation=True, max_length=256)
        return pd.DataFrame({"hf_label": [o["label"] for o in out], "hf_score": [o["score"] for o in out]})

    pred_hf = (pred_propio
               .withColumn("hf", sentimiento_hf("review"))
               .select("app_id", "review", "p_propio", "veredicto_propio",
                       col("hf.hf_label").alias("veredicto_hf"), col("hf.hf_score").alias("p_hf")))
    pred_hf.persist().count()

In [ ]:
if USAR_HUGGINGFACE:
    comparacion = pred_hf.withColumn(
        "veredicto_hf_bool", col("veredicto_hf") == "POSITIVE"
    ).withColumn("de_acuerdo", col("veredicto_propio") == col("veredicto_hf_bool"))

    acuerdo = comparacion.agg(F.avg(col("de_acuerdo").cast("double"))).first()[0]
    print(f"Acuerdo entre modelo propio y DistilBERT: {acuerdo:.1%}")

    print("\nEjemplos donde DIFIEREN (los interesantes para discutir en clase):")
    (comparacion.filter(~col("de_acuerdo"))
     .select("review", "p_propio", "veredicto_hf", "p_hf")
     .show(10, truncate=70))

**Para discutir en el cierre:** cuando no coinciden, ¿a cuál le creen más? El modelo
propio vio *solo* reseñas de Steam — aprendió el vocabulario específico de esta
comunidad (jerga de videojuegos, sarcasmo tipo "10/10 would crash again"). DistilBERT
vio muchísimo más texto en general, pero nada de Steam en particular. Ninguno de los
dos es "la verdad" — `recomienda` tampoco lo es, es la opinión de una sola persona.

## Para practicar (y para cerrar el curso)

1. **`ALPHA_ALS` y `BOOST`/`DAMPEN`:** ninguno de los tres se ajustó a este dataset
   específico — son puntos de partida de la literatura. Prueba una rejilla y compara
   Precision@10.
2. **Otro modelo de Hugging Face:** busca en el Hub uno entrenado específicamente en
   reseñas de producto o de videojuegos (si existe uno público) y compara contra
   DistilBERT — ¿cambia el % de acuerdo con el modelo propio?
3. **Cold start real:** ¿qué le recomendarías a alguien sin ninguna reseña? (pista: la
   línea base de "top global" de este notebook ya es ese fallback).
4. **De los tres recomendadores del curso** (anime explícito, Steam implícito) — ¿cuál
   fue más fácil de evaluar honestamente, y por qué?

Gracias por el semestre — nos vemos en el siguiente dataset.

In [ ]:
spark.stop()